# POLSCI 3

## Week 4, Lecture Notebook 2: Omitted Variable/Selection Bias

Here, we'll continue using the wellness data from earlier this week---but this time we'll be using the **real** data. 

This means we won't be able to see both potential outcomes for everyone.

We will only observe the outcome that actually happened, given the treatment a subject actually got.

### Revisiting the Data

Let's read in the wellness dataset and recap what each of the variables represent. 

In [ ]:
# Do not edit this cell, just run it.
library(testthat)
wellness <- read.csv('ps3_wellness_real.csv')
head(wellness)

This dataset contains real data on participants in a wellness program. Each row represents a unique respondent, and the data measures medical expenditure before and after the wellness treatment. Here is more information about the variables:

`id`: Respondent ID (anonymized identifier for each respondent)

`treat`: Whether or not the treatment **was offered** to the participant (treatment here is the wellness program) (`1` = was offered wellness program, `0` = wasn't offered wellness program). This is randomly assigned.

`participate`: Whether respondent **actually participated** in the wellness program (`1` = actually participated, `0` = didn't participate)

`baseline`: Monthly average medical costs at baseline; that is, before the program started.

`outcome_post`: Monthly cost of medical care for this person after the workplace wellness program started (regardless of whether they participated or not).

## Random assignment eliminates confounding

Starting next week, we're going to talk much more about how to analyze experiments.  

But as a teaser, note the following.  In a randomized experiment, the random assignment of units to treatment or control groups gets rid of the problem of confounding---i.e., it eliminates selection bias and omitted variable bias.

It would be a problem for inferring the causal effect of the program if less healthy people tended to be assigned to the wellness program.  

After all, we might expect unhealthy people to have higher monthly costs of medical care. So we might expect the medical costs in the treatment group to be higher than the control group, just because unhealthy people tend to be assigned to treatment. So this might bias against the wellness program---that is, make it look less effective than it really is. 

But if instead a flip of a coin---i.e., random assignment---determines whether <code>treat=1</code> or <code>treat=0</code>, then unhealthy people are as likely to go into the control group as the treatment group.

This should take care of the problem of confounding. In a randomized experiment, the subjects don't get to decide whether they are assigned to treatment or not---so there is no self selection. And there should be no third variables (that is, "omitted variables") linked to treatment assignment and the outcome variable.

## Analyzing the experiment the right way 

So how should we analyze the experimental data?  The simplest and most compelling method is simply to compare outcomes in the two groups created by the randomization. 

In R, we could do this the way we have been doing it so far: 

- create a subset of the units for whom <code>treat=1</code> and calculate the average monthly cost of medical care after the program started (i.e., `outcome_post`);
- do the same for the subset of the units for whom <code>treat=0</code>; and then
- compare the two averages to see if they differ.

(In a few weeks, we'll talk about how to distinguish real differences from differences we might expect to be produced just by the luck of the draw---i.e., the chance error introduced by random assignment).

The following code implements this analysis:

In [ ]:
# Calculate the average outcome in the group with treat=1 
treated <- subset(wellness, treat==1) # subset to the treatment group
treated_ave_cost <- mean(treated$outcome_post) # take the average of the outcome (monthly medical costs)
treated_ave_cost # print the result



In [ ]:
# Calculate the average outcome in the group with treat=0
control <- subset(wellness, treat==0) # subset to the treatment group
control_ave_cost <- mean(control$outcome_post) # print the result
control_ave_cost

In [ ]:
# Now calculate the difference of means between the groups
treated_ave_cost - control_ave_cost 

This doesn't look like a big difference in average monthly costs---which seems like evidence against the effectiveness of the program. 

## Time out for <code>difference_in_means()</code>

Before we go on, let's learn a different function in R that makes it easier to do the calculation above. 

We'll use this function a lot, so this is a good chance to learn it. 

This function, <code>difference_in_means()</code>, is built into an R package that is already loaded in your Data Hub.  

All you need to do is to call the library that loads the function (this library is called <code>estimatr</code>; this step will often be done for you in assignments). 

Then, for the syntax: what goes inside the parentheses is an expression like this:

<code>difference_in_means(outcome_variable ~ treatment_variable, dataset_name)</code>



In [ ]:
library(estimatr)
difference_in_means(outcome_post ~ treat, wellness)

Note that this code prints out a lot of extra stuff that we don't need yet, like standard errors, t-values, and the like.  

We'll be talking more what this stuff is over the next few weeks.

For now, just look at the first number (the one labelled "Estimate") and verify that it is the same as the difference we calculated "manually" above---i.e., -28.5596.

Indeed, this is just the difference of means (also called "difference in means") -- i.e., it is the average outcome in the treatment group, minus the average outcome in the control group.

You will run exactly the analysis above in your in-class assignment on Thursday.

## The problem of selection bias

While we discussed the right way to analyze the experiment above---that is, by comparing the groups created by the random assignment---this dataset also gives us the chance to look at the problem of self-selection and the problem of confounding.

Notice that the dataset records another variable called <code>participate</code>.  This variable records whether a subject **actually participated** in the wellness program (`1` = actually participated, `0` = didn't participate).

This variable is not randomly assigned.  After all, as a researcher I can tell you that you were randomly selected to be eligible to participate in a workness wellness program.  But I can't actually force you to take advantage of any of the services offered through the program...



### Groups in the dataset

Thus, here the researchers first randomized people to have the opportunity to participate in the wellness program or not. That's what they can control as researchers, and it's what makes this study a randomized experiment.

However, among the people who had the chance to participate, some chose to do so and some chose not to do so. 

This means there are three groups of people:

1. `treat == 0`, people assigned to the control group and who therefore didn't have the chance to participate (so `participate == 0` for all of them, too)
2. `treat == 1` and `participate == 0`, people who had the opportunity to participate and chose **not** to
3. `treat == 1` and `participate == 1`, people who had the opportunity to participate and **did** choose to

Someone who wants to understand the effect of the wellness program might want to compare groups 2 and 3. After all, they might reason, the `participate == 1` group took part in the wellness program while the `participate == 0` did not.  So can't we compare the two groups to assess the impact of actual participation?

This is not a good idea, however, because of the problem of self-selection and the confounding it can introduce.  After all, as groups, those who opt to participate might be different than those who don't.  In particular, they might care more about their health---and thus have lower medical costs anyway.  

Or---following the example in the section "Random Assignment Eliminates Confounding" above---sicker people might choose to participate, so we might expect them to have higher medical costs after the program, whether they participate or not.  

Either way, this analysis effectively turns the randomized experiment into an observational study---in which it is the subjects rather than the researcher who decide whether to receive the "treatment."  

We can expect that the **potential outcomes** of those in the `participate == 1` group are sytematically different from the **potential outcomes** of those in the `participate == 0` group 

While we can sometimes learn a lot from observational studies, this does introduce an additional complication for our interpretation and assessment of the program's impact.

Let's see what happens if analyze the data this (wrong) way.  To do this analysis, create two new subsets in the dataset:

- a subset called `chose.not.to.participate` that includes group 2 above: the people who 1) were assigned to the treatment group, and so had a chance to participate, but 2) chose not to
- a subset called `chose.to.participate` that includes group 3 above: the people who 1) were assigned to the treatment group and 2) *did* choose to participate


In the below cell, let's make those subsets.  

Here, I am also going to use a new shortcut in R---the final new R technique for this week.  

Using the techniques from past weeks, we would 
- create a subset of those in the treatment group (<code>treat == 1 </code>) in the dataset <code>wellness</code> and save it a different R dataset; and then
- create subsets in that new dataset of those who participated (<code>participate == 1 </code>) or did not participate (<code>participate == 0 </code>).

It turns out though that we can do these steps all at once in R, using the <code>&</code> command in between the two variables we are using to do the subsetting -- here, <code>treat</code> and <code>participate</code>.

Here is an example:

In [ ]:
# Create the two subsets

chose.not.to.participate <- subset(wellness, treat == 1 & participate == 0) 
chose.to.participate <- subset(wellness, treat == 1 & participate == 1)

Now, let's use the strategy of comparing outcomes (medical costs after the wellness program started) among those who choose to participate in the program to those who choose not to participate in the program.

First, let's take the mean of the outcome variable (`outcome_post`) among those in the <code>treat == 1</code> group who chose not to participate (using the `chose.not.to.participate` subset) and print the result.


In [ ]:
# Take the average spending of those in the treat==1 group who chose not to participate in the program
mean.spending.non.participants <- mean(chose.not.to.participate$outcome_post) 
mean.spending.non.participants # print the result


Now, let's take the average of later medical spending for those who chose to participate in the program, that is, the mean of `outcome_post` in `chose.to.participate`.

In [ ]:
# Take the average spending of those in the treat==1 group who did choose to participate in the program
mean.spending.participants <- mean(chose.to.participate$outcome_post) 
mean.spending.participants # print the result

This is a big difference from our analysis of the randomized groups above!  

Here, wellness program participants spent 525 dollars on medical care after the program started. 

People who did not participate spent 650 dollars on medical care after the program started. 

Thus, on average, wellness program participants spent 125 dollars less on medical care than did non-participants in the months after the program.

It looks like the program was effective!  

But this is very likely because of self-selection: among those assigned to the treatment group, people who chose to participate in the program are different than those who chose not to do so.

# Comparing the self-selected groups on a baseline variable

In fact, we can do a little more data analysis to see if there is evidence of this problem, that the self-selected groups were different from each other to begin with.

Remember that the dataset contains the `baseline` variable. This measures how much people spent on medical care **before** the wellness program began. 

So, let's compare how much the people who *later* decided to or not to participate in the wellness program spent on medical care *before* the wellness program even started.

First, let's take the average of the `baseline` variable among the self-selected **non-participators** (the `chose.not.to.participate` subset)

In [ ]:
# take the mean of baseline in chose.not.to.participate 
mean.baseline.spending.non.participants <- mean(chose.not.to.participate$baseline) 
mean.baseline.spending.non.participants



Now, take the average of the `baseline` variable among the self-selected **participators** (the `chose.to.participate` subset)

In [ ]:
# take the mean of `baseline` in `chose.to.participate`.
mean.baseline.spending.participants <- mean(chose.to.participate$baseline) # SOLUTION
mean.baseline.spending.participants

In [ ]:
mean.baseline.spending.non.participants - mean.baseline.spending.participants

Thus, on average, in the assigned-to-treatment group, wellness program **participants** spent 424 dollars on medical care before the program started. 

On average in the assigned-to-treatment group, people who **did not participate** spent 540 dollars on medical care before the program started. 

So wellness program participants on average spent 115 dollars per month less on medical care before the wellness program started.

Remember, this cannot be evidence of a causal effect -- we are looking at differences between these groups **before** they were offered the chance to participate in the program.

This suggests that the difference ex-post (i.e., after the program) between those who chose to participate and those who did not is that the kind of people who choose to participate in the wellness program are healthier to begin with.  

There seems to be some kind of bias from self selection, due to unobserved confounders that differ between the two self-selected groups.

Note: we will be making concepts like "bias" and what we mean by it more precise in coming weeks.  For now, just think of it as something that implies that a comparison between two groups is not a good guide to true differences, e.g., a good guide to the causal impact of some treatment status that differs across the groups.

Overall, it looks like we can reach the same conclusions as <a href="https://academic.oup.com/qje/article/134/4/1747/5550759?login=true" target="_blank">the study</a>:

> We find strong patterns of selection: during the year prior to the intervention, program participants had lower medical expenditures and healthier behaviors than nonparticipants. ... we do not find significant causal effects of treatment on total medical expenditures.

## What we learned in this Week 4's Notebook Lecture 2:

- How random assignment limits confounding (and we had a first discussion of how to analyze experimental data correctly);
- How the problem of confounding can arise from selection bias or omitted variables bias---even in an experiment, if the data are not analyzed the correct way; and
- Two new techniques in R:
   - How to use the <code>difference-in-means()</code> function, available in the <code>estimatr</code> library; and
   - How to subset on more than one condition using <code>&</code> 